# FIT5196 Assessment 1 - EDA Notebook

**Group:** Group050  
**Component owner:** Jason — Figures 3–4, Findings 4–5 and MLQ-2  

This template-formatted component is ready for integration into the shared `Group050_EDA.ipynb`. Other numbered figures, findings and MLQs remain assigned to their respective owners.


## 0. Configuration and data loading

Load the six standardised CSVs produced by the solution notebook. Keep paths
relative/configurable and use explicit read options where the literal string
`NaN` must remain visible.


In [ ]:
from pathlib import Path

GROUP_ID = "Group050"
OUTPUT_DIR = Path("outputs")


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display


# Task 5 reads the six standardised CSVs only. It does not parse or clean the
# original JSON/XML again. keep_default_na=False preserves literal "NaN".
TABLE_NAMES = [
    "orders",
    "order_items",
    "customers",
    "deliveries",
    "products",
    "product_reviews",
]

table_paths = {
    table_name: OUTPUT_DIR / f"{GROUP_ID}_{table_name}_standardised.csv"
    for table_name in TABLE_NAMES
}
missing_paths = [
    str(path) for path in table_paths.values() if not path.is_file()
]
if missing_paths:
    raise FileNotFoundError(
        "Run the solution notebook first or place all six standardised CSVs "
        f"in outputs/: {missing_paths}"
    )

standardised_tables = {
    table_name: pd.read_csv(
        path,
        keep_default_na=False,
        encoding="utf-8",
    )
    for table_name, path in table_paths.items()
}

orders = standardised_tables["orders"].copy()
customers = standardised_tables["customers"].copy()
orders["order_id"] = orders["order_id"].astype("string")
orders["customer_id"] = orders["customer_id"].astype("string")
customers["customer_id"] = customers["customer_id"].astype("string")
orders["order_total"] = pd.to_numeric(orders["order_total"], errors="raise")
orders["order_timestamp"] = pd.to_datetime(
    orders["order_timestamp"],
    format="%Y-%m-%d %H:%M:%S",
    errors="raise",
)

if not orders["order_id"].is_unique:
    raise AssertionError("orders.order_id must be unique before EDA joins.")
if not customers["customer_id"].is_unique:
    raise AssertionError(
        "customers.customer_id must be unique for a many-to-one join."
    )

orders_before_join = len(orders)
unique_orders_before_join = orders["order_id"].nunique()
customer_order_eda = orders.merge(
    customers[["customer_id", "customer_segment", "loyalty_tier"]],
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)
orders_after_join = len(customer_order_eda)
unique_orders_after_join = customer_order_eda["order_id"].nunique()
unmatched_customer_rows = int(
    customer_order_eda["_merge"].ne("both").sum()
)
join_multiplication_passed = (
    orders_before_join == orders_after_join
    and unique_orders_before_join == unique_orders_after_join
    and unmatched_customer_rows == 0
)
if not join_multiplication_passed:
    raise AssertionError(
        "Customer join changed the order grain or left unmatched IDs."
    )
customer_order_eda = customer_order_eda.drop(columns="_merge")

eda_join_audit = pd.DataFrame(
    [
        {
            "join": "orders.customer_id -> customers.customer_id",
            "relationship": "many_to_one",
            "rows_before": orders_before_join,
            "rows_after": orders_after_join,
            "unique_orders_before": unique_orders_before_join,
            "unique_orders_after": unique_orders_after_join,
            "unmatched_customer_rows": unmatched_customer_rows,
            "status": "PASS" if join_multiplication_passed else "FAIL",
        }
    ]
)
display(eda_join_audit)


## 1. Context and data-preparation assurance

Briefly define the business context and data scope. Summarise 3-5 material
transformation decisions and 4-6 material validation results by citing stable
`MAP-...` and `VAL-...` IDs. Do not repeat the complete mapping or notebook.


The analysis covers the six canonical retail tables produced by the solution notebook, while this component uses the `orders` and `customers` tables for customer evidence. Stable IDs preserve case and leading zeroes (`MAP-customers-01`), signup dates use `YYYY-MM-DD` (`MAP-customers-02`), and booleans remain `True`/`False` (`MAP-customers-10`). The Task 4 register confirmed exact schemas (`VAL-SCHEMA-01`–`06`), published types (`VAL-TYPE-01`–`06`), no empty or Python/pandas missing outputs (`VAL-MISSING-01`–`06`), a complete unique customer key (`VAL-PK-CUSTOMERS-01`), resolved customer foreign keys (`VAL-FK-CUSTOMERS-01`), and complete source-key coverage (`VAL-FLOW-01`–`06`). For this EDA join, `validate="many_to_one"` retains 5,000 rows and 5,000 unique orders with no unmatched customers.


## 2. Assessed EDA visualisations

Submit 6-8 clearly labelled assessed figures. Across the set, cover all six
published categories, at least four tables and at least two valid relational
analyses. For each figure state the question, observation unit, denominator,
tables/join keys, interpretation and material limitation.


### Figure 1: Univariate distribution or composition

**Owner/status:** Reserved for the orders/order-items owner and to be integrated into the shared notebook.


In [ ]:
# Assigned team member inserts Figure 1 here.


### Figure 2: Bivariate relationship or group comparison

**Owner/status:** Reserved for the orders/order-items owner and to be integrated into the shared notebook.


In [ ]:
# Assigned team member inserts Figure 2 here.


### Figure 3: Order value by customer segment

**Question:** How does canonical order value differ across customer segments?  
**Observation unit and denominator:** One canonical order; all 5,000 orders with a resolved customer profile, with each order contributing once.  
**Tables and join keys:** `orders` and `customers`; `orders.customer_id = customers.customer_id`. The validated many-to-one join retained 5,000 rows and 5,000 unique `order_id` values, with no unmatched customers.  
**Interpretation and limitation:** Mainstream had the highest median order total (AUD 2,561.43), while Small Business had the lowest (AUD 2,440.52), a difference of AUD 120.91 or approximately 5.0%. The strongly overlapping distributions show that within-segment variability is much greater than the between-segment median difference. This is descriptive, not causal; product mix, customer tenure and order frequency may explain part of the difference.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter


figure3_summary = (
    customer_order_eda.groupby("customer_segment", observed=True)
    .agg(
        order_count=("order_id", "nunique"),
        mean_order_total=("order_total", "mean"),
        median_order_total=("order_total", "median"),
        q1_order_total=("order_total", lambda values: values.quantile(0.25)),
        q3_order_total=("order_total", lambda values: values.quantile(0.75)),
    )
    .sort_values("median_order_total", ascending=False)
)

segment_order = figure3_summary.index.tolist()
segment_values = [
    customer_order_eda.loc[
        customer_order_eda["customer_segment"].eq(segment),
        "order_total",
    ].to_numpy()
    for segment in segment_order
]
segment_labels = [
    f"{segment}\n(n={int(figure3_summary.loc[segment, 'order_count']):,})"
    for segment in segment_order
]

fig, ax = plt.subplots(figsize=(10.5, 6.2))
boxplot = ax.boxplot(
    segment_values,
    labels=segment_labels,
    patch_artist=True,
    showfliers=True,
    medianprops={"color": "#17324D", "linewidth": 2.2},
    boxprops={"linewidth": 1.2},
    whiskerprops={"linewidth": 1.1},
    capprops={"linewidth": 1.1},
    flierprops={
        "marker": "o",
        "markersize": 2.5,
        "markerfacecolor": "#526D82",
        "markeredgecolor": "none",
        "alpha": 0.30,
    },
)

palette = ["#5B8FF9", "#61DDAA", "#F6BD16", "#65789B"]
for patch, colour in zip(boxplot["boxes"], palette):
    patch.set_facecolor(colour)
    patch.set_alpha(0.72)

ax.set_title(
    "Figure 3. Order value distribution by customer segment",
    loc="left",
    fontsize=14,
    fontweight="bold",
    pad=14,
)
ax.set_xlabel("Customer segment (canonical order count)")
ax.set_ylabel("Order total (AUD)")
ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"${value:,.0f}"))
ax.grid(axis="y", alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()

figure3_path = OUTPUT_DIR / "Figure_3_order_value_by_customer_segment.png"
fig.savefig(figure3_path, dpi=180, bbox_inches="tight")
plt.show()

highest_segment = figure3_summary["median_order_total"].idxmax()
lowest_segment = figure3_summary["median_order_total"].idxmin()
highest_median = figure3_summary.loc[highest_segment, "median_order_total"]
lowest_median = figure3_summary.loc[lowest_segment, "median_order_total"]
median_gap = highest_median - lowest_median
relative_gap = median_gap / lowest_median * 100

display(figure3_summary.round(2))

print("FIGURE_3_COMPLETE rows=5000; join_multiplication_check=PASS")


### Figure 4: Monthly revenue pattern by customer segment

**Question:** How did canonical order revenue change over time, and did the pattern differ by customer segment?  
**Observation unit and denominator:** One customer-segment/month revenue aggregate; all 5,000 canonical orders allocated once to their order month and customer segment. Revenue is not normalised by customer count.  
**Tables and join keys:** `orders` and `customers`; `orders.customer_id = customers.customer_id`. The monthly aggregation reconciles to the original 5,000 orders and total canonical revenue.  
**Interpretation and limitation:** Segment revenue peaks occurred in different months: Mainstream in October (AUD 381,761.91), Premium in July (AUD 388,351.87), Small Business in April (AUD 331,844.89), and Value in November (AUD 361,477.66). This suggests non-uniform timing across segments. However, only one historical year is observed, and totals reflect segment size and order frequency, so recurring seasonality cannot be established.


In [ ]:
customer_order_eda["order_month"] = (
    customer_order_eda["order_timestamp"].dt.to_period("M").dt.to_timestamp()
)

figure4_monthly = (
    customer_order_eda.groupby(
        ["order_month", "customer_segment"],
        as_index=False,
        observed=True,
    )
    .agg(
        monthly_revenue=("order_total", "sum"),
        order_count=("order_id", "nunique"),
    )
    .sort_values(["order_month", "customer_segment"])
)

# A second aggregation-level multiplication check ensures every order and its
# revenue are represented exactly once after grouping.
figure4_order_count_check = int(figure4_monthly["order_count"].sum())
figure4_revenue_check = float(figure4_monthly["monthly_revenue"].sum())
if figure4_order_count_check != len(customer_order_eda):
    raise AssertionError("Monthly segment grouping did not preserve order count.")
if not np.isclose(
    figure4_revenue_check,
    customer_order_eda["order_total"].sum(),
    atol=0.01,
):
    raise AssertionError("Monthly segment grouping did not preserve revenue.")

fig, ax = plt.subplots(figsize=(11.0, 6.4))
for colour, segment in zip(palette, segment_order):
    segment_monthly = figure4_monthly.loc[
        figure4_monthly["customer_segment"].eq(segment)
    ]
    ax.plot(
        segment_monthly["order_month"],
        segment_monthly["monthly_revenue"] / 1_000_000,
        label=segment,
        color=colour,
        marker="o",
        linewidth=2.0,
        markersize=4.5,
    )

ax.set_title(
    "Figure 4. Monthly order revenue by customer segment",
    loc="left",
    fontsize=14,
    fontweight="bold",
    pad=14,
)
ax.set_xlabel("Order month")
ax.set_ylabel("Monthly order revenue (AUD millions)")
ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"${value:.2f}M"))
ax.grid(alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(title="Customer segment", frameon=False, ncol=2)
fig.autofmt_xdate(rotation=0)
fig.tight_layout()

figure4_path = OUTPUT_DIR / "Figure_4_monthly_revenue_by_customer_segment.png"
fig.savefig(figure4_path, dpi=180, bbox_inches="tight")
plt.show()

peak_rows = (
    figure4_monthly.loc[
        figure4_monthly.groupby("customer_segment")[
            "monthly_revenue"
        ].idxmax()
    ]
    .sort_values("customer_segment")
    .reset_index(drop=True)
)
peak_descriptions = "; ".join(
    f"{row.customer_segment}: {row.order_month:%b %Y} "
    f"(${row.monthly_revenue:,.0f})"
    for row in peak_rows.itertuples(index=False)
)

display(figure4_monthly.round({"monthly_revenue": 2}))

print("FIGURE_4_COMPLETE orders=5000; aggregation_reconciliation=PASS")


### Figure 5: Review or text behaviour

**Owner/status:** Reserved for the products/product-reviews owner and to be integrated into the shared notebook.


In [ ]:
# Assigned team member inserts Figure 5 here.


### Figure 6: Delivery or operational performance

**Owner/status:** Reserved for the deliveries owner and to be integrated into the shared notebook.


In [ ]:
# Assigned team member inserts Figure 6 here.


### Optional Figure 7–8

Reserved for coordinated additional figures from the assigned team members. Delete this section if the final group notebook contains only six assessed figures.


In [ ]:
# Assigned team members insert optional Figure 7–8 here.


## 3. Ten evidence-based findings

Write exactly ten numbered findings. Keep each concise and decision-focused.
Each must identify the evidence, grain, magnitude/denominator, business meaning,
an alternative explanation or uncertainty, and a proportionate implication.
One figure may support more than one genuinely distinct finding.


1. **Finding 1:** Reserved for the orders/order-items owner.
2. **Finding 2:** Reserved for the orders/order-items owner.
3. **Finding 3:** Reserved for the orders/order-items owner.
4. **Finding 4 — Customer segments have similar typical order values:** Figure 3 compares one canonical order at a time across all 5,000 matched orders. Mainstream customers had the highest median order total at AUD 2,561.43, while Small Business customers had the lowest at AUD 2,440.52—a difference of only AUD 120.91, or 5.0% of the lower median. Their interquartile ranges also overlapped substantially (AUD 1,475.71–4,019.82 and AUD 1,383.03–3,926.96 respectively). Segment membership alone therefore provides limited separation in order value. Product mix, tenure or order frequency could instead explain the gap, so differentiated offers should be tested with behavioural indicators before broad deployment.
5. **Finding 5 — Monthly revenue peaks differ by customer segment:** Figure 4 aggregates the 5,000 canonical orders into 48 segment/month observations. Revenue peaks were not synchronised: Mainstream peaked in October at AUD 381,761.91, Premium in July at AUD 388,351.87, Small Business in April at AUD 331,844.89, and Value in November at AUD 361,477.66. The timing difference may support segment-specific campaign planning, but totals reflect segment size and order frequency, and only one year is observed. One-off promotions or random variation remain plausible. Treat segment timing as a hypothesis and validate it using additional years or controlled campaigns.
6. **Finding 6:** Reserved for the deliveries owner.
7. **Finding 7:** Reserved for the deliveries owner.
8. **Finding 8:** Reserved for the deliveries owner.
9. **Finding 9:** Reserved for the products/product-reviews owner.
10. **Finding 10:** Reserved for the products/product-reviews owner.


## 4. Five future machine-learning questions

Write exactly five numbered questions covering at least two problem types.
Model training is not required. Keep each response compact enough for the
ten-page report.


### MLQ-1: Reserved for the assigned owner

| Element | Response |
|---|---|
| Integration status | To be completed and merged by the assigned team member. |


### MLQ-2: How much will each customer spend next month?

| Element | Response |
|---|---|
| EDA evidence | Figures 3–4 show overlapping order-value distributions but different revenue peaks, suggesting that behaviour and timing add information beyond the published segment label. |
| Business decision | Estimate future customer value for retention planning and segment-level budgeting. |
| Problem type and analysis unit | Regression; one customer at a month-end snapshot. |
| Target or unsupervised objective | Total `order_total` generated by that customer during the following calendar month. |
| Decision-time predictors | Historical order count, recency, average order value, prior-12-month orders, lifetime value, segment, loyalty tier, preferred channel and recent channel mix, all measured at the snapshot. |
| Validation split and metric | Rolling-origin validation, training only on earlier months; report MAE and RMSE to show typical and large errors. |
| Leakage, fairness or deployment risk | Future orders, updated lifetime value and post-cutoff fields cause temporal leakage. Sparse histories, behavioural drift and unequal errors across customer segments require monitoring. |


### MLQ-3: Reserved for the assigned owner

| Element | Response |
|---|---|
| Integration status | To be completed and merged by the assigned team member. |


### MLQ-4: Reserved for the assigned owner

| Element | Response |
|---|---|
| Integration status | To be completed and merged by the assigned team member. |


### MLQ-5: Reserved for the assigned owner

| Element | Response |
|---|---|
| Integration status | To be completed and merged by the assigned team member. |


## 5. Limitations and conclusion

Summarise the most decision-relevant limitations, what the data cannot establish
and the next evidence needed. End with a concise conclusion rather than new
analysis.


For the customer component, the main limitations are the single observed year, descriptive rather than causal comparisons, and segment totals that combine customer population, order frequency and order value. The evidence does not establish recurring seasonality or prove that segment membership causes spending differences. Additional years, campaign exposure data and product-mix controls are needed. Within those limits, the results support testing behaviour-aware customer strategies and segment-specific timing rather than assuming that the published segments have sharply different order values.


## References

- Group050 standardised `orders` and `customers` CSV files produced by the solution notebook.
- `public_data_dictionary.csv`, FIT5196 Assessment 1 supplied data contract.
- `Group050_source_to_target_mapping.csv`, Group050 transformation lineage.
